In [1]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7"
}

url = "https://www.aravot.am/category/news/politics/"
r = requests.get(url, headers=headers)
soup = BeautifulSoup(r.text, "html.parser")

links = soup.select("a[href*='aravot.am']")

article_links = []
for a in links:
    href = a.get("href")
    if href and re.search(r"/\d{4}/\d{2}/\d{2}/\d+/?$", href):
        article_links.append(href)

article_links = list(set(article_links))
print("Статей найдено:", len(article_links))


def get_article_text(url):
    r = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")

    content_div = soup.find("div", class_="entry-content")
    if not content_div:
        content_div = soup.find("div", class_="post-content")

    if content_div:
        paragraphs = content_div.find_all("p")
    else:
        paragraphs = soup.find_all("p")

    text = " ".join(p.get_text(" ", strip=True) for p in paragraphs)
    return text



data = []

for article_url in article_links[:20]:
    try:
        text = get_article_text(article_url)
        if len(text) > 50:
            data.append({
                "category": "politics",
                "text": text
            })
        time.sleep(0.5)
    except Exception as e:
        print("Ошибка:", e)

df = pd.DataFrame(data)
print(df.shape)
df.head()

Статей найдено: 53
(20, 2)


,category,text
0,politics,-Նախորդ դատական նիստում մեղադրողը չի կատարել դ...
1,politics,Region Monitor . Թուրքիայի նախագահ Ռեջեփ Թայիփ...
2,politics,ՀՀ հակակոռուպցիոն դատարանում շարունակվում է Գյ...
3,politics,Բաքվի զինվորական դատարանում տեղի է ունենում բա...
4,politics,«Ես ցանկություն չունեմ մեկնաբանելու տարբեր միջ...


In [4]:
data = []

for article_url in article_links[:10]:
    try:
        text = get_article_text(article_url)
        print("URL:", article_url)
        print("Длина текста:", len(text))

        if len(text) > 50:
            data.append({
                "category": "politics",
                "text": text
            })

        time.sleep(0.5)
    except Exception as e:
        print("Ошибка:", e)

df = pd.DataFrame(data)
print("DF shape:", df.shape)
print(df.head())

URL: https://www.aravot.am/2026/02/05/1537727/
Длина текста: 1703
URL: https://www.aravot.am/2026/02/05/1537650/
Длина текста: 604
URL: https://www.aravot.am/2026/02/05/1537696/
Длина текста: 3856
URL: https://www.aravot.am/2026/02/05/1537653/
Длина текста: 528
URL: https://www.aravot.am/2026/02/05/1537731/
Длина текста: 1827
URL: https://www.aravot.am/2026/02/05/1537693/
Длина текста: 746
URL: https://www.aravot.am/2026/02/05/1537746/
Длина текста: 2813
URL: https://www.aravot.am/2026/02/05/1537728/
Длина текста: 1191
URL: https://www.aravot.am/2026/02/05/1537759/
Длина текста: 1578
URL: https://www.aravot.am/2026/02/05/1537747/
Длина текста: 926
DF shape: (10, 2)
   category                                               text
0  politics  -Նախորդ դատական նիստում մեղադրողը չի կատարել դ...
1  politics  Region Monitor . Թուրքիայի նախագահ Ռեջեփ Թայիփ...
2  politics  ՀՀ հակակոռուպցիոն դատարանում շարունակվում է Գյ...
3  politics  Բաքվի զինվորական դատարանում տեղի է ունենում բա...
4  politics

In [5]:
categories = {
    "politics": "https://www.aravot.am/category/news/politics/",
    "rights": "https://www.aravot.am/category/news/rights/",
    "society": "https://www.aravot.am/category/news/society/",
    "culture": "https://www.aravot.am/category/news/culture/",
    "education": "https://www.aravot.am/category/news/education/",
    "sport": "https://www.aravot.am/category/news/sport/",
    "press": "https://www.aravot.am/category/news/press/"
}

In [6]:
all_data = []

for category, base_url in categories.items():
    print(f"\nПарсим категорию: {category}")

    for page in range(1, 26):
        page_url = base_url if page == 1 else f"{base_url}page/{page}/"
        print("Страница:", page_url)

        r = requests.get(page_url, headers=headers)
        soup = BeautifulSoup(r.text, "html.parser")

        links = soup.select("a[href*='aravot.am']")
        article_links = []

        for a in links:
            href = a.get("href")
            if href and re.search(r"/\d{4}/\d{2}/\d{2}/\d+/?$", href):
                article_links.append(href)

        article_links = list(set(article_links))
        print("Найдено статей:", len(article_links))

        for article_url in article_links:
            try:
                text = get_article_text(article_url)

                if len(text) > 50:
                    all_data.append({
                        "category": category,
                        "text": text
                    })

                time.sleep(0.1)
            except Exception as e:
                print("Ошибка:", e)


Парсим категорию: politics
Страница: https://www.aravot.am/category/news/politics/
Найдено статей: 53
Страница: https://www.aravot.am/category/news/politics/page/2/
Найдено статей: 53
Страница: https://www.aravot.am/category/news/politics/page/3/
Найдено статей: 60
Страница: https://www.aravot.am/category/news/politics/page/4/
Найдено статей: 61
Страница: https://www.aravot.am/category/news/politics/page/5/
Найдено статей: 63
Страница: https://www.aravot.am/category/news/politics/page/6/
Найдено статей: 63
Страница: https://www.aravot.am/category/news/politics/page/7/
Найдено статей: 63
Страница: https://www.aravot.am/category/news/politics/page/8/
Найдено статей: 63


KeyboardInterrupt: 

In [ ]:
df = pd.DataFrame(all_data)
print(df.shape)

df.to_csv("aravot_news_dataset2.csv", index=False)

In [ ]:
df

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^ա-ևa-zа-яё\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["text"].apply(clean_text)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
df.to_csv("aravot_news_dataset_clean2.csv", index=False)

In [ ]:
df

In [ ]:
df['category'].unique()